# Discovering Climate Teleconnections with Machine Learning

**Welcome!** In this notebook, we'll explore how different parts of Earth's climate system are connected across vast distances - a phenomenon called *teleconnections*.

## What are Teleconnections?

Have you ever wondered why weather patterns in one part of the world can affect climate thousands of kilometers away? For example:

- **El Nino** in the Pacific Ocean affects rainfall in Africa and temperatures in Europe
- **Arctic sea ice** changes can influence winter weather in North America
- The **Atlantic Multidecadal Oscillation (AMO)** affects hurricane activity and European summers

These are teleconnections - statistical relationships between climate variables at distant locations.

## What We'll Do

1. **Explore** climate data from 1000-year climate simulations
2. **Train** machine learning models to discover which climate indices predict others
3. **Visualize** the teleconnection patterns we find

Let's begin!

---
## Part 1: Setting Up Our Environment

First, let's configure matplotlib for non-interactive (headless) rendering, set up the working directory, and check our computing environment.

In [ ]:
# Configure matplotlib for non-interactive use (must be done before any other matplotlib imports)
import matplotlib
matplotlib.use('Agg')

# Set working directory to the notebook's own directory
import os
os.chdir(os.path.dirname(os.path.abspath('__file__')) if os.path.exists('__file__') else os.getcwd())

print(f"Working directory: {os.getcwd()}")

In [ ]:
# Check if we have GPU acceleration available
import subprocess

gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)

if gpu_result.returncode == 0:
    print(f"GPU Available: {gpu_result.stdout.strip()}")
    print("Training will be accelerated!")
else:
    print("No GPU detected - we'll use CPU (slower but works fine)")

In [ ]:
# Import our analysis tools
import sys
import warnings
warnings.filterwarnings('ignore')  # Keep output clean

# Add our custom modules
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts', 'lrbased_teleconnection'))

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Handle optional dependencies gracefully
try:
    import xgboost as xgb
    print("xgboost: available")
except ImportError:
    print("xgboost: not available (XGBoost models will be skipped)")

try:
    import pycwt
    print("pycwt: available")
except ImportError:
    print("pycwt: not available (wavelet filtering will be skipped)")

# Our teleconnection analysis modules
from dataloader import load_data, preprocess_data, generate_lagdata
from models import ModelSelector
from main import main as run_training

# Metrics (used later for manual analysis)
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

print("\nAll tools loaded successfully!")

---
## Part 2: Exploring the Climate Data

Our data comes from **NorESM** (Norwegian Earth System Model) - a sophisticated climate model that simulates Earth's climate over 1000 years.

We have three simulation scenarios:
- **slow**: Historical forcing with slow volcanic/solar variations (850-2005 AD)
- **shigh**: Historical forcing with high volcanic/solar variations
- **picntrl**: Pre-industrial control (stable conditions for 1000 years)

Let's load and explore the data!

In [ ]:
# Load the climate dataset
data_file = os.path.join(os.getcwd(), 'dataset', 'noresm-f-p1000_slow_new_jfm.csv')
data = pd.read_csv(data_file)

print(f"Dataset shape: {data.shape[0]} years x {data.shape[1]} variables")
print(f"\nTime period: {data['Time'].min()} to {data['Time'].max()} AD")
print(f"\nClimate variables ({data.shape[1]-1} total):")

# Show the variables in a readable format
variables = data.columns[1:].tolist()
for i in range(0, len(variables), 5):
    print("  " + ", ".join(variables[i:i+5]))

In [ ]:
# Preview the first few rows
print("First 5 rows of data:")
data.head()

### Understanding the Variables

Each column represents a climate index. Here are some key ones:

| Variable | What it measures |
|----------|------------------|
| `amo1`, `amo2`, `amo3` | Atlantic Multidecadal Oscillation (different definitions) |
| `ensoSSTjfm` | El Nino Southern Oscillation (Pacific temperatures) |
| `naoPSLjfm` | North Atlantic Oscillation (pressure patterns) |
| `nhSICjfm` | Northern Hemisphere Sea Ice Concentration |
| `AMOCann` | Atlantic Meridional Overturning Circulation |
| `glTSjfm` | Global Temperature |

The suffix `jfm` means January-February-March average.

---
## Part 3: Visualization

Let's visualize some key climate indices over the 1000-year simulation period to understand the variability in the data.

In [ ]:
# 4-panel climate index plots
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

indices_to_plot = [
    ('amo2', 'Atlantic Multidecadal Oscillation', 'steelblue'),
    ('ensoSSTjfm', 'El Nino (Pacific SST)', 'coral'),
    ('naoPSLjfm', 'North Atlantic Oscillation', 'forestgreen'),
    ('nhSICjfm', 'Arctic Sea Ice', 'purple')
]

for ax, (var, title, color) in zip(axes.flat, indices_to_plot):
    ax.plot(data['Time'], data[var], color=color, alpha=0.7, linewidth=0.8)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Year')
    ax.grid(True, alpha=0.3)

plt.suptitle('1000 Years of Climate Variability', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('climate_indices_overview.png', dpi=100, bbox_inches='tight')
plt.show()

print("Notice the variability! Our goal is to find which indices can predict others.")

---
## Part 4: The Machine Learning Methodology

### The Key Question

**Can we predict future values of one climate index using past values of other indices?**

For example: Can sea ice concentration 20 years ago help predict today's Atlantic temperatures?

### Our Method: Lagged Features

The concept of **lagged features** is central to our approach:

1. **Choose a target** (e.g., `amo2` - Atlantic oscillation)
2. **Create lagged features** - shift all other climate indices back in time by various amounts (e.g., 15-25 years)
3. **Train a model** to predict the target from these lagged features
4. **Evaluate** using correlation and error metrics
5. **Identify** which lagged features are most important (= teleconnections!)

For instance, if `20_lag_ensoSSTjfm` is selected as an important feature, it means **El Nino conditions 20 years ago** help predict today's Atlantic temperatures.

### Interpreting Results

**Correlation Score**: How well the predictions match reality (0 = no match, 1 = perfect)
- Above 0.7 = Strong teleconnection discovered!
- 0.5-0.7 = Moderate relationship
- Below 0.5 = Weak or no teleconnection

**MAE (Mean Absolute Error)**: Average prediction error on a 0-100 normalized scale
- Below 12 = Good prediction accuracy
- 12-20 = Moderate accuracy
- Above 20 = Poor predictions

In [ ]:
# Demonstrate the lagged features concept
target = 'amo2'  # What we want to predict

# Preprocess the data (normalize to 0-100 scale)
data_processed = load_data(data_file, [], year_start=850)
data_processed = preprocess_data(data_processed, target)

print(f"Target variable: {target}")
print(f"\nData normalized to 0-100 scale")
print(f"Example values for {target}: min={data_processed[target].min():.1f}, max={data_processed[target].max():.1f}")
print(f"\nNumber of time steps: {len(data_processed)}")
print(f"Number of predictor variables: {len(data_processed.columns) - 2}")  # minus Time and target

# Show how lagged data looks
lag_demo = generate_lagdata(15, 25, data_processed, target, with_mean_feature=False)
print(f"\nAfter creating lagged features (lag 15-25):")
print(f"  Feature matrix shape: {lag_demo.shape}")
print(f"  Example lagged feature names: {[c for c in lag_demo.columns if 'lag' in c][:5]}")

---
## Part 5: Quick Experiment

Now let's train a model! We'll use a simple **Linear Regression** to establish a baseline.

**Parameters:**
- `target_feature='amo2'`: Predicting the Atlantic Multidecadal Oscillation
- `max_allowed_features=6`: Use top 6 predictors
- `end_lag=50`: Look back up to 50 years
- `step_lag=10`: 10-year lag windows

In [ ]:
# Run a quick experiment
import time

print("Training a Linear Regression model to predict AMO...")
print("This will test different lag windows to find the best predictors.\n")

t_start = time.time()

result_file = run_training(
    data_file=data_file,
    target_feature='amo2',
    delete_features=[],
    modelname='LinearRegression',
    max_allowed_features=6,      # Use top 6 predictors
    end_lag=50,                   # Look back up to 50 years
    n_ensembles=1,               # Linear regression is deterministic
    splitsize=0.6,               # 60% training, 40% testing
    with_mean_feature=True,
    main_year_start=850,
    step_lag=10,                 # 10-year lag windows
    with_wavelet_filter=False,
    is_jupyter_run=True
)

elapsed = time.time() - t_start
print(f"\nExperiment completed in {elapsed:.1f} seconds")
print(f"Results saved to: {result_file}")

---
## Part 6: Results Analysis

Let's load the CSV results we just generated and examine the best model performance.

In [ ]:
# Load and examine the results
results = pd.read_csv(result_file)

print("Experiment Results:")
print("=" * 60)
print(f"Total lag steps evaluated: {len(results)}")
print(f"Correlation range: {results['corr_score'].min():.3f} to {results['corr_score'].max():.3f}")
print(f"MAE range: {results['mae_score'].min():.2f} to {results['mae_score'].max():.2f}")

# Show the best result
best_idx = results['corr_score'].abs().idxmax()
best = results.loc[best_idx]

print(f"\nBest Model Performance:")
print(f"  Correlation Score: {best['corr_score']:.3f}")
print(f"  Mean Absolute Error: {best['mae_score']:.2f}")
print(f"  Forecast Horizon: {best['max_forecast']:.0f} years")
print(f"  Max Lag Used: {best['max_lag']:.0f} years")
print(f"\nTop Predictors (teleconnections found):")

# Parse the selected features
features = best['selected_features'].split(', ')
for i, feat in enumerate(features[:6], 1):
    if '_lag_' in feat:
        parts = feat.split('_lag_')
        print(f"  {i}. {parts[1]} (lag: {parts[0]} years)")

# Show all results as a table
print(f"\nAll lag steps:")
print(results[['max_lag', 'init_lag', 'max_forecast', 'corr_score', 'mae_score']].to_string(index=False))

---
## Part 7: Feature Importance Visualization

Which climate indices from the past are most important for predicting our target? Let's visualize the feature importances from the best model.

In [ ]:
# Feature importance bar plot from the best result
import ast

# Get feature names and importances from the best result
feature_names = ast.literal_eval(best['selected_feature_list'])
feature_importances = ast.literal_eval(best['feature_importances'])

# Create readable labels
labels = []
for fname in feature_names:
    if '_lag_' in fname:
        parts = fname.split('_lag_')
        labels.append(f"{parts[1]}\n(lag {parts[0]}y)")
    else:
        labels.append(fname)

# Sort by importance
sorted_pairs = sorted(zip(labels, feature_importances), key=lambda x: abs(x[1]), reverse=True)
sorted_labels, sorted_importances = zip(*sorted_pairs)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_labels)))
bars = ax.barh(range(len(sorted_labels)), [abs(x) for x in sorted_importances], color=colors)
ax.set_yticks(range(len(sorted_labels)))
ax.set_yticklabels(sorted_labels, fontsize=9)
ax.set_xlabel('Feature Importance (|coefficient|)', fontsize=11)
ax.set_title(f'Top Predictors of amo2 (Best Model, corr={best["corr_score"]:.3f})', fontsize=13)
ax.invert_yaxis()  # Most important at top
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("Higher bars indicate stronger teleconnection signals.")
print("The lag value shows how many years in the past the predictor comes from.")

---
## Part 8: Multi-Target Comparison

Let's run the same experiment for 2-3 different target variables and compare how well each can be predicted. This reveals which climate indices have the strongest teleconnection signals.

In [ ]:
# Run experiments for multiple target variables
comparison_targets = ['amo2', 'ensoSSTjfm', 'naoPSLjfm']
comparison_results = []

print(f"Running experiments for {len(comparison_targets)} target variables...")
print(f"Parameters: end_lag=50, step_lag=10, max_features=6, LinearRegression\n")

for target_name in comparison_targets:
    t_start = time.time()
    print(f"  Training for target: {target_name}...", end=" ")
    
    try:
        res_file = run_training(
            data_file=data_file,
            target_feature=target_name,
            delete_features=[],
            modelname='LinearRegression',
            max_allowed_features=6,
            end_lag=50,
            n_ensembles=1,
            splitsize=0.6,
            with_mean_feature=True,
            main_year_start=850,
            step_lag=10,
            with_wavelet_filter=False,
            is_jupyter_run=True
        )
        
        df = pd.read_csv(res_file)
        best_row = df.loc[df['corr_score'].abs().idxmax()]
        elapsed = time.time() - t_start
        
        comparison_results.append({
            'Target': target_name,
            'Best Correlation': f"{abs(best_row['corr_score']):.3f}",
            'Best MAE': f"{best_row['mae_score']:.2f}",
            'Forecast (years)': f"{best_row['max_forecast']:.0f}",
            'Time (s)': f"{elapsed:.1f}",
        })
        
        print(f"done ({elapsed:.1f}s) - corr={abs(best_row['corr_score']):.3f}")
        
    except Exception as e:
        print(f"FAILED: {e}")
        comparison_results.append({
            'Target': target_name,
            'Best Correlation': 'ERROR',
            'Best MAE': 'ERROR',
            'Forecast (years)': '-',
            'Time (s)': '-',
        })

# Display comparison table
print("\n" + "=" * 70)
print("MULTI-TARGET COMPARISON")
print("=" * 70)
comparison_df = pd.DataFrame(comparison_results)
print(comparison_df.to_string(index=False))

print("\nInterpretation:")
print("  - Higher correlation = stronger teleconnection signals detected")
print("  - Lower MAE = more accurate predictions")
print("  - Forecast years = how far ahead the model can predict")

---
## Part 9: Summary

### Key Takeaways

1. **Teleconnections are real**: Climate indices in different regions are statistically connected across decades

2. **ML can discover them**: By using lagged features, we can find which past conditions predict future climate states

3. **Forecast horizons vary**: Some teleconnections work for 10-year predictions, others for 30+ years

4. **Analysis workflow**:
   - Run experiments to generate results
   - Filter by quality (correlation > 0.7, MAE < 12)
   - Visualize with `plot_climate_index_analysis()` for overview
   - Deep-dive with `combined_plots()` for specific targets

### What to Explore Next

- Try different target variables to find new teleconnections
- Compare models (Random Forest often finds different patterns than Linear Regression)
- Increase `end_lag` to discover longer-range teleconnections
- Use the pre-industrial control dataset to test if patterns hold without external forcing
- Enable wavelet filtering to focus on specific frequency bands

### Resources

- **Documentation**: https://naicno.github.io/wp7-UC1-climate-indices-teleconnection/
- **Repository**: https://github.com/NAICNO/wp7-UC1-climate-indices-teleconnection
- **Reference Paper**: Omrani et al. (2022) - NPJ Climate and Atmospheric Science

In [ ]:
# Final check: show generated result files
import glob

gen3_results = glob.glob(os.path.join(os.getcwd(), 'results', 'gen3.0', '**', '*.csv'), recursive=True)
print(f"Total result files in results/gen3.0/: {len(gen3_results)}")

if gen3_results:
    print("\nLatest results:")
    for r in sorted(gen3_results)[-5:]:
        # Show path relative to project root
        rel = os.path.relpath(r, os.getcwd())
        print(f"  {rel}")

print("\nNotebook execution complete.")